# EEG Preprocessing with MNE-Python
## Workload Classification: Time-Window Extraction [-1.5, +1.5]s

This notebook demonstrates preprocessing of EEG data using MNE-Python, extracting time-locked epochs around workload-related markers with a 3-second window centered on each event.

## 1. Import Required Libraries

Import necessary libraries including MNE, NumPy, Pandas, and Matplotlib for EEG preprocessing and visualization.

In [ ]:
import os
import sys
import pyxdf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from collections import Counter

# Set MNE logging level
mne.set_log_level('warning')

print(f"MNE version: {mne.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load XDF Data and Identify Workload Markers

Load the XDF file and filter markers to extract only workload-related events, regardless of task/no-task, sparkles, count, or other attributes.

In [ ]:
# Load XDF file
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
xdf_file = os.path.join(data_dir, 'sub-PD089', 'ses-S001', 'eeg', 'sub-PD089_ses-S001_task-demo_workload_run-001_eeg.xdf')

print(f"Loading XDF file: {xdf_file}")
streams, header = pyxdf.load_xdf(xdf_file)
print(f"Loaded {len(streams)} streams\n")

# Extract EEG stream (index 4) and marker stream (index 5)
EEG_STREAM_INDEX = 4
MARKER_STREAM_INDEX = 5

eeg_stream = streams[EEG_STREAM_INDEX]
marker_stream = streams[MARKER_STREAM_INDEX]

# Extract raw data
eeg_data = np.array(eeg_stream['time_series'])
eeg_timestamps = np.array(eeg_stream['time_stamps'])
marker_data = marker_stream['time_series']
marker_timestamps = marker_stream['time_stamps']
sampling_rate = float(eeg_stream['info']['nominal_srate'][0])

print(f"EEG shape: {eeg_data.shape}")
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Total markers: {len(marker_data)}\n")

In [ ]:
# Flatten marker data and identify all unique markers
try:
    flat_markers = [item[0] if isinstance(item, (list, tuple)) else item for item in marker_data]
except:
    flat_markers = marker_data

# Count marker frequencies
marker_counts = Counter(flat_markers)

print("--- All Markers in Dataset ---")
for marker, count in marker_counts.most_common():
    print(f"{marker:<40} | Count: {count:4d}")

# Filter for workload markers only
SKIP_MARKERS = {'Recording/Start', 'Recording/End'}
TARGET_MARKER_PREFIX = 'workload'

workload_markers = []
workload_timestamps = []

for marker, marker_ts in zip(flat_markers, marker_timestamps):
    if marker in SKIP_MARKERS:
        continue
    if TARGET_MARKER_PREFIX.lower() in marker.lower():
        workload_markers.append(marker)
        workload_timestamps.append(marker_ts)

print(f"\n--- Workload Markers (Target Events) ---")
print(f"Total workload events: {len(workload_markers)}")
print(f"Unique workload types: {len(set(workload_markers))}")
print(f"\nWorkload marker distribution:")
for marker, count in Counter(workload_markers).most_common():
    print(f"  {marker:<40} | Count: {count:4d}")

## 3. Create MNE Raw Object

Construct an MNE Raw object from the EEG time series data with proper channel information and sampling frequency.

In [ ]:
# Extract channel labels
try:
    eeg_info = eeg_stream['info']
    channel_labels = [ch['label'][0] for ch in eeg_info['desc'][0]['channels'][0]['channel']]
except:
    channel_labels = [f"CH{i}" for i in range(eeg_data.shape[1])]

print(f"Channel count: {len(channel_labels)}")
print(f"First 5 channels: {channel_labels[:5]}")
print(f"Last 5 channels: {channel_labels[-5:]}\n")

# Create MNE channel info
ch_types = ['eeg'] * len(channel_labels)
info = mne.create_info(ch_names=channel_labels, sfreq=sampling_rate, ch_types=ch_types)

# Create Raw array (MNE expects shape (n_channels, n_samples))
raw = mne.io.RawArray(eeg_data.T, info, verbose=False)

print(f"Created MNE Raw object:")
print(f"  Shape: {raw.get_data().shape}")
print(f"  Duration: {raw.times[-1]:.2f} seconds ({raw.times[-1]/60:.2f} minutes)")
print(raw)

## 4. Extract and Annotate Events from Markers

Convert marker timestamps and labels into MNE events format (sample indices and event IDs). Create a mapping of workload marker types to numeric event IDs.

In [ ]:
# Create MNE events array from workload markers
events = []
event_id_dict = {}
current_event_id = 1

for marker, marker_ts in zip(workload_markers, workload_timestamps):
    # Find closest sample index to marker timestamp
    sample_idx = np.searchsorted(eeg_timestamps, marker_ts)
    
    # Create unique event ID for each marker type
    if marker not in event_id_dict:
        event_id_dict[marker] = current_event_id
        current_event_id += 1
    
    # events array format: [sample_index, 0, event_id]
    events.append([sample_idx, 0, event_id_dict[marker]])

events = np.array(events, dtype=int)

print(f"Created events array:")
print(f"  Shape: {events.shape}")
print(f"  Sample range: [{events[:, 0].min()}, {events[:, 0].max()}]")
print(f"\nEvent ID mapping (Workload markers):")
for marker, event_id in sorted(event_id_dict.items(), key=lambda x: x[1]):
    print(f"  ID {event_id}: {marker}")

## 5. Define and Create Epochs

Define epochs with a time window of [-1.5, +1.5] seconds around each workload marker using mne.Epochs().

In [ ]:
# Define epoch time window: [-1.5, +1.5] seconds
TMIN = -1.5
TMAX = 1.5

print(f"Creating epochs: [{TMIN}, {TMAX}] seconds around workload markers\n")

# Create epochs using MNE
epochs = mne.Epochs(
    raw, events, event_id_dict,
    tmin=TMIN, tmax=TMAX,
    baseline=None,  # No baseline correction yet
    preload=True,   # Load all data into memory
    reject_by_annotation=False,
    verbose=False
)

print(f"Epochs created successfully:")
print(f"  Shape: {epochs.get_data().shape}")  # (n_epochs, n_channels, n_samples)
print(f"  Number of epochs: {len(epochs)}")
print(f"  Time points per epoch: {epochs.get_data().shape[2]}")
print(f"  Duration per epoch: {(TMAX - TMIN):.2f} seconds\n")

# Summary statistics
print("Epoch distribution by marker:")
for event_id, marker in sorted([(v, k) for k, v in event_id_dict.items()]):
    indices = np.where(epochs.events[:, 2] == event_id)[0]
    print(f"  {marker:<40} | Count: {len(indices):4d}")

## 6. Apply Preprocessing Filters

Apply high-pass and low-pass filters to remove noise and artifacts. Also apply notch filtering to remove line noise.

In [ ]:
# Apply filters to epochs
print("Applying preprocessing filters to epochs...\n")

# High-pass filter (remove slow drifts and DC offset)
l_freq = 0.5  # Hz
print(f"Applying high-pass filter: {l_freq} Hz")
epochs.filter(l_freq=l_freq, h_freq=None, verbose=False)

# Low-pass filter (remove high-frequency noise)
h_freq = 50.0  # Hz
print(f"Applying low-pass filter: {h_freq} Hz")
epochs.filter(l_freq=None, h_freq=h_freq, verbose=False)

# Notch filter to remove line noise (50 Hz for Europe, 60 Hz for US)
line_freq = 50.0  # Hz (European standard)
print(f"Applying notch filter: {line_freq} Hz (line noise)")
epochs.notch_filter(freqs=line_freq, verbose=False)

print("\nFilters applied successfully!")

# Apply common average reference (CAR)
print("Applying common average reference (CAR)...")
epochs.set_eeg_reference('average', verbose=False)

print("Preprocessing complete!")

## 7. Validate and Inspect Preprocessed Epochs

Inspect the preprocessed epochs using MNE plotting functions and display statistics.

In [ ]:
# Display epochs information
print("="*60)
print("PREPROCESSED EPOCHS SUMMARY")
print("="*60)
print(f"\nEpochs object:")
print(f"  Shape: {epochs.get_data().shape}")
print(f"  Data type: {epochs.get_data().dtype}")
print(f"  Memory size: {epochs.get_data().nbytes / (1024**2):.2f} MB")

# Get statistics per channel
epochs_data = epochs.get_data()
print(f"\nData statistics (all epochs, all channels):")
print(f"  Min: {epochs_data.min():.4f} µV")
print(f"  Max: {epochs_data.max():.4f} µV")
print(f"  Mean: {epochs_data.mean():.4f} µV")
print(f"  Std: {epochs_data.std():.4f} µV")

# Check for any remaining artifacts or bad epochs
print(f"\nEpoch status:")
print(f"  Total epochs: {len(epochs)}")
print(f"  Good epochs: {len(epochs) - len(epochs.drop_log)}")
print(f"  Bad epochs: {len(epochs.drop_log)}")

In [ ]:
# Visualize sample epochs (first 3 epochs)
fig = epochs[:3].plot(picks='all', scalings='auto', show=True)
plt.suptitle("Sample Preprocessed Epochs (First 3 workload events)", fontsize=14, y=1.00)

In [ ]:
# Plot average epochs per workload condition
fig, axes = plt.subplots(1, len(event_id_dict), figsize=(15, 4))
if len(event_id_dict) == 1:
    axes = [axes]

for idx, (marker, event_id) in enumerate(sorted(event_id_dict.items(), key=lambda x: x[1])):
    # Get epochs for this condition
    condition_epochs = epochs[marker]
    avg_response = condition_epochs.average()
    avg_response.plot(axes=axes[idx], show=False)
    axes[idx].set_title(f"Average: {marker}\n(n={len(condition_epochs)})")

plt.tight_layout()
plt.show()

## 8. Save Preprocessed Data

Save the preprocessed epochs in MNE format (.fif) and also convert to NumPy for use with REVE model.

In [ ]:
# Create output directory
output_dir = os.path.join(os.getcwd(), 'eeg_processing', 'output')
os.makedirs(output_dir, exist_ok=True)

# Save epochs in MNE format
filename_fif = os.path.join(output_dir, 'sub-PD089_workload_epochs.fif')
print(f"Saving epochs to: {filename_fif}")
epochs.save(filename_fif, overwrite=True)

# Save as NumPy arrays for REVE processing
epochs_data = epochs.get_data()  # Shape: (n_epochs, n_channels, n_samples)
labels_array = epochs.events[:, 2]  # Event IDs

filename_epochs = os.path.join(output_dir, 'sub-PD089_workload_epochs.npy')
filename_labels = os.path.join(output_dir, 'sub-PD089_workload_labels.npy')

print(f"Saving epochs array to: {filename_epochs}")
np.save(filename_epochs, epochs_data)

print(f"Saving labels to: {filename_labels}")
np.save(filename_labels, labels_array)

# Create a DataFrame with metadata
metadata_df = pd.DataFrame({
    'epoch_id': np.arange(len(epochs)),
    'event_id': labels_array,
    'marker': [id_to_marker[eid] for eid in labels_array],
    'sample_index': epochs.events[:, 0]
})

filename_metadata = os.path.join(output_dir, 'sub-PD089_workload_metadata.csv')
print(f"Saving metadata to: {filename_metadata}")
metadata_df.to_csv(filename_metadata, index=False)

print("\n✓ All files saved successfully!")
print(f"\nOutput files:")
print(f"  - {filename_fif}")
print(f"  - {filename_epochs}")
print(f"  - {filename_labels}")
print(f"  - {filename_metadata}")

In [ ]:
# Create reverse mapping for readable output
id_to_marker = {v: k for k, v in event_id_dict.items()}

print("\n" + "="*60)
print("READY FOR REVE FEATURE EXTRACTION")
print("="*60)
print(f"\nPreprocessed data summary:")
print(f"  Shape: {epochs_data.shape}")
print(f"  - Epochs: {epochs_data.shape[0]}")
print(f"  - Channels: {epochs_data.shape[1]}")
print(f"  - Samples per epoch: {epochs_data.shape[2]}")
print(f"  - Time window: [{TMIN:.1f}, {TMAX:.1f}] seconds")
print(f"  - Sampling rate: {sampling_rate} Hz")
print(f"\nClass distribution:")
for event_id, marker in sorted([(v, k) for k, v in event_id_dict.items()]):
    count = np.sum(labels_array == event_id)
    percentage = 100 * count / len(labels_array)
    print(f"  {marker:<40} | {count:4d} ({percentage:5.1f}%)")

print(f"\nNext steps:")
print(f"1. Use eeg_processing.main.process_subject() to extract REVE features")
print(f"2. Or load the saved NumPy arrays for custom processing")
print(f"3. Use REVE model for classification or feature extraction")